In [ ]:
# =========================
# Core
# =========================
import os
import numpy as np
import pandas as pd

# =========================
# Visualization
# =========================
import matplotlib.pyplot as plt
import seaborn as sns

# =========================
# Scikit-learn (preprocessing + metrics)
# =========================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.metrics import classification_report, precision_recall_curve, average_precision_score

# =========================
# Handling imbalanced data
# =========================
from sklearn.utils import class_weight

# =========================
# Deep Learning (Keras)
# =========================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

# =========================
# Callbacks (recommended)
# =========================
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
import kagglehub

# Download latest version
#path = kagglehub.dataset_download("nelgiriyewithana/credit-card-fraud-detection-dataset-2023")

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)

In [ ]:
#check for dataset path
os.listdir(path)

In [ ]:
#Create de dataframe
file_path = os.path.join(path, 'creditcard.csv')
df = pd.read_csv(file_path)

#explore the data distribution 
df.head()
df.info()
df.describe()

In [ ]:
#check for target class values and distribution
df["Class"].value_counts()
df["Class"].value_counts(normalize=True)

In [ ]:
#Create a test-train split for baseline
X = df.drop(columns=["Time", "Class"])
y = df["Class"]

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify = y
)

y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)

In [ ]:
#scale features to train a NN
scaler = StandardScaler()

scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.mean(axis=0)
X_train_scaled.std(axis=0)
X_train_scaled.shape

In [ ]:
#Define de model using subclassing
class Detector(keras.Model):
    def __init__(self):
        super().__init__()
        
        self.dense1 = layers.Dense(64, activation='relu')
        self.dropout1 = layers.Dropout(0.3)

        self.dense2 = layers.Dense(32, activation='relu')
        self.dropout2 = layers.Dropout(0.3)

        self.output_layer = layers.Dense(1,activation='sigmoid')

    def call(self,inputs) :
        x = self.dense1(inputs)
        x = self.dropout1(x)

        x = self.dense2(x)
        x = self.dropout2(x)

        return self.output_layer(x)

In [ ]:
#Compile the model
model = Detector()

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(curve='PR', name='pr_auc')
    ]
)

In [ ]:
#train the model with the data and apply earlystopping and class weight after the V2.0

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = {0: weights[0], 1: weights[1]}

early_stop = EarlyStopping(
    monitor='val_pr_auc',
    mode='max',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
y_prob = model.predict(X_test_scaled)

print("min:", y_prob.min())
print("max:", y_prob.max())
print("mean:", y_prob.mean())

print("fraud rate real:", y_test.mean())

In [ ]:
threshold = 0.5

y_pred = (y_prob > threshold).astype(int)

In [ ]:
print(classification_report(y_test, y_pred))
ap = average_precision_score(y_test, y_prob)
print("PR-AUC:", ap)
plt.hist(y_prob, bins=50)
plt.title("Predicted probabilities")
plt.show()

In [ ]:
# Loss
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title("Loss")
plt.show()

# Recall
plt.plot(history.history['recall'], label='train_recall')
plt.plot(history.history['val_recall'], label='val_recall')
plt.legend()
plt.title("Recall")
plt.show()

# Precision
plt.plot(history.history['precision'], label='train_precision')
plt.plot(history.history['val_precision'], label='val_precision')
plt.legend()
plt.title("Precision")
plt.show()